# Statistical NLP Exercises

Eleven short exercises covering the statistical foundations of NLP: frequency counts,
vocabulary, co-occurrence, sparsity, smoothing, bigram prediction, TF-IDF, vector
similarity, and conditional probability. Each section states the objective, then works
through it in code, then prints the result.

**Author:** Vishal Sigdel

## Exercise 1: Word Frequency Counter

Count how many times each word appears in a sentence — the simplest form of term
frequency (TF). Split on whitespace, tally into a dictionary.

In [1]:
text = "I like NLP I like Python NLP is fun"

words = text.split()
freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1

for w, c in freq.items():
    print(f"{w}: {c}")

I: 2
like: 2
NLP: 2
Python: 1
is: 1
fun: 1


## Exercise 2: Vocabulary Builder

Find all unique words across a corpus of sentences. A `set` naturally deduplicates, and
its size is the vocabulary size — a number that matters later for smoothing.

In [2]:
corpus = [
    "I like NLP",
    "I like AI",
    "AI is amazing",
]

vocabulary = set()
for sentence in corpus:
    vocabulary.update(sentence.split())

print("Vocabulary")
for w in sorted(vocabulary):
    print(w)
print(f"\nVocabulary Size = {len(vocabulary)}")

Vocabulary
AI
I
NLP
amazing
is
like

Vocabulary Size = 6


## Exercise 3: Word Co-occurrence Matrix

Build a co-occurrence matrix with window size 1: for each word, count which words appear
immediately before or after it across the sentence. This is the basis for distributional
word representations — words that share contexts end up with similar rows.

In [3]:
sentence = "the cat sat on the mat"
tokens = sentence.split()
vocab = sorted(set(tokens))
window = 2

co_matrix = {w1: {w2: 0 for w2 in vocab} for w1 in vocab}

for i, word in enumerate(tokens):
    for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
        if i != j:
            co_matrix[word][tokens[j]] += 1

In [4]:
import pandas as pd
co_df = pd.DataFrame(co_matrix).T[vocab]
co_df

,cat,mat,on,sat,the
cat,0,0,1,1,1
mat,0,0,1,0,1
on,1,1,0,1,1
sat,1,0,1,0,2
the,1,1,1,2,0


## Exercise 4: Detect Data Sparsity

Most word pairs never co-occur — real corpora produce mostly-zero co-occurrence
matrices. This traverses the matrix from Exercise 3 and lists the empty cells, the
signature of a **sparse matrix**.

In [4]:
never_occurred = []
for w1 in vocab:
    for w2 in vocab:
        if w1 != w2 and co_matrix[w1][w2] == 0:
            never_occurred.append((w1, w2))

for pair in never_occurred:
    print(f"{pair} -> Never occurred")

('cat', 'mat') -> Never occurred
('cat', 'on') -> Never occurred
('mat', 'cat') -> Never occurred
('mat', 'on') -> Never occurred
('mat', 'sat') -> Never occurred
('on', 'cat') -> Never occurred
('on', 'mat') -> Never occurred
('sat', 'mat') -> Never occurred
('sat', 'the') -> Never occurred
('the', 'sat') -> Never occurred


## Exercise 5: Laplace Smoothing

Raw counts assign zero probability to anything unseen, which breaks downstream math
(zero probabilities dominate products). Laplace (add-one) smoothing fixes this by adding
1 to every count before normalising:

$$
P(w) = \frac{\text{Count}(w) + 1}{\text{Total} + \text{Vocabulary}}
$$

In [5]:
word_counts = {"cat": 4, "dog": 3, "bird": 1}

total = sum(word_counts.values())
vocab_size = len(word_counts)

for word, count in word_counts.items():
    prob = (count + 1) / (total + vocab_size)
    print(f"{word} : {prob:.2f}")

cat : 0.45
dog : 0.36
bird : 0.18


## Exercise 6: Simple Keyboard Prediction

Build bigram counts from a training corpus, then predict the most likely next word after
a given word — the same idea behind phone keyboard suggestions. Ranks candidates by raw
frequency and shows the top 3.

In [5]:
training_corpus = [
    "I like NLP",
    "I like Python",
    "I like coffee",
    "I love NLP",
]

bigram_counts = {}
for sentence in training_corpus:
    tokens = sentence.split()
    for w1, w2 in zip(tokens, tokens[1:]):
        bigram_counts.setdefault(w1, {})
        bigram_counts[w1][w2] = bigram_counts[w1].get(w2, 0) + 1

def top_n_predictions(word, n=3):
    candidates = bigram_counts.get(word, {})
    ranked = sorted(candidates.items(), key=lambda kv: kv[1], reverse=True)
    return [w for w, _ in ranked[:n]]

query = "like"
print("Suggestions")
for w in top_n_predictions(query):
    print(w)

Suggestions
NLP
Python
coffee


## Exercise 7: TF-IDF

Term frequency alone over-weights common words. TF-IDF discounts words that appear in
many documents (low information) and boosts words specific to a document:

- **TF**: how often a term appears in one document
- **DF**: in how many documents a term appears
- **IDF**: inverse of DF — rare-across-corpus terms score higher
- **TF-IDF** = TF × IDF

In [6]:
import math

documents = [
    "I like NLP",
    "I like AI",
    "AI is amazing",
]

tokenized_docs = [doc.split() for doc in documents]
vocab_tfidf = sorted(set(w for doc in tokenized_docs for w in doc))
n_docs = len(documents)

def tf(term, doc_tokens):
    return doc_tokens.count(term) / len(doc_tokens)

def df(term):
    return sum(1 for doc in tokenized_docs if term in doc)

def idf(term):
    return math.log(n_docs / df(term))

rows = []
for i, doc_tokens in enumerate(tokenized_docs):
    for term in vocab_tfidf:
        if term in doc_tokens:
            rows.append({
                "doc": f"Document {i + 1}",
                "term": term,
                "TF": round(tf(term, doc_tokens), 3),
                "DF": df(term),
                "IDF": round(idf(term), 3),
                "TF-IDF": round(tf(term, doc_tokens) * idf(term), 3),
            })

In [7]:
import pandas as pd
pd.DataFrame(rows)

,doc,term,TF,DF,IDF,TF-IDF
0,Document 1,I,0.333,2,0.405,0.135
1,Document 1,NLP,0.333,1,1.099,0.366
2,Document 1,like,0.333,2,0.405,0.135
3,Document 2,AI,0.333,2,0.405,0.135
4,Document 2,I,0.333,2,0.405,0.135
5,Document 2,like,0.333,2,0.405,0.135
6,Document 3,AI,0.333,2,0.405,0.135
7,Document 3,amazing,0.333,1,1.099,0.366
8,Document 3,is,0.333,1,1.099,0.366


## Exercise 8: Dense Vector Similarity

Word embeddings are dense vectors (unlike sparse co-occurrence rows). Cosine similarity
measures how aligned two vectors are, independent of magnitude — the standard way to
compare embeddings.

In [8]:
import math

def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    mag1 = math.sqrt(sum(a * a for a in v1))
    mag2 = math.sqrt(sum(b * b for b in v2))
    return dot / (mag1 * mag2)

cat = [0.4, 0.2, 0.8]
dog = [0.5, 0.1, 0.7]

similarity = cosine_similarity(cat, dog)
print(f"Cosine Similarity = {similarity:.2f}")

Cosine Similarity = 0.98


## Exercise 9: Find Similar Words

Extend cosine similarity to a small embedding table: compare one target word's vector
against every other word and return the closest match — the core operation behind
"words most similar to X" queries.

In [9]:
embeddings = {
    "cat": [0.4, 0.2, 0.8],
    "dog": [0.5, 0.1, 0.7],
    "car": [-0.7, 0.8, -0.2],
    "lion": [0.45, 0.25, 0.82],
}

def most_similar(target, embeddings):
    target_vec = embeddings[target]
    best_word, best_score = None, -1
    for word, vec in embeddings.items():
        if word == target:
            continue
        score = cosine_similarity(target_vec, vec)
        if score > best_score:
            best_word, best_score = word, score
    return best_word, best_score

word, score = most_similar("cat", embeddings)
print(f"Most similar to cat\n{word}\nSimilarity = {score:.2f}")

Most similar to cat
lion
Similarity = 1.00


## Exercise 10 (Assignment): Mini NLP Pipeline

Combines Exercises 1–6 into one pipeline over a new corpus: vocabulary, frequency, TF,
co-occurrence, sparsity, Laplace-smoothed unigram probabilities, and bigram-based next-word
prediction — the small end-to-end system the earlier exercises were building toward.

In [10]:
pipeline_corpus = [
    "I love NLP",
    "I love AI",
    "AI loves Python",
    "Python loves data",
]

# 1. Vocabulary
pipeline_vocab = set()
for sentence in pipeline_corpus:
    pipeline_vocab.update(sentence.split())
print(f"Vocabulary Size : {len(pipeline_vocab)}")

# 2. Word frequencies
pipeline_freq = {}
for sentence in pipeline_corpus:
    for w in sentence.split():
        pipeline_freq[w] = pipeline_freq.get(w, 0) + 1

# 3. TF (per corpus, pooled)
total_pipeline_words = sum(pipeline_freq.values())
pipeline_tf = {w: c / total_pipeline_words for w, c in pipeline_freq.items()}

top_words = sorted(pipeline_freq.items(), key=lambda kv: kv[1], reverse=True)[:3]
print("Top Words")
for w, c in top_words:
    print(f"{w} : {c}")

# 4. Co-occurrence matrix (window size 1, across whole corpus)
pipeline_tokens_by_sentence = [s.split() for s in pipeline_corpus]
pipeline_co = {w1: {w2: 0 for w2 in pipeline_vocab} for w1 in pipeline_vocab}
for tokens in pipeline_tokens_by_sentence:
    for i, word in enumerate(tokens):
        if i > 0:
            pipeline_co[word][tokens[i - 1]] += 1
        if i < len(tokens) - 1:
            pipeline_co[word][tokens[i + 1]] += 1

# 5. Sparse pairs
pipeline_sparse = [
    (w1, w2)
    for w1 in pipeline_vocab
    for w2 in pipeline_vocab
    if w1 != w2 and pipeline_co[w1][w2] == 0
]

# 6. Laplace-smoothed unigram probabilities
pipeline_vocab_size = len(pipeline_vocab)
pipeline_smoothed = {
    w: (c + 1) / (total_pipeline_words + pipeline_vocab_size)
    for w, c in pipeline_freq.items()
}

# 7. Bigram counts + next-word prediction
pipeline_bigrams = {}
for tokens in pipeline_tokens_by_sentence:
    for w1, w2 in zip(tokens, tokens[1:]):
        pipeline_bigrams.setdefault(w1, {})
        pipeline_bigrams[w1][w2] = pipeline_bigrams[w1].get(w2, 0) + 1

def pipeline_predict(word, n=3):
    candidates = pipeline_bigrams.get(word, {})
    ranked = sorted(candidates.items(), key=lambda kv: kv[1], reverse=True)
    return [w for w, _ in ranked[:n]]

query_word = "love"
print(f"\nPrediction after \"{query_word}\"")
for w in pipeline_predict(query_word):
    print(w)

print("\nSparse Pairs")
for pair in pipeline_sparse[:3]:
    print(pair)

Vocabulary Size : 7
Top Words
I : 2
love : 2
AI : 2

Prediction after "love"
NLP
AI

Sparse Pairs
('loves', 'I')
('loves', 'NLP')
('loves', 'love')


## Exercise 11: Predict the Next Word Using Conditional Probability

Formalises Exercise 6's prediction with actual conditional probabilities rather than raw
counts:

$$P(w_2 \\mid w_1) = \\frac{Count(w_1, w_2)}{Count(w_1)}$$

Builds bigram counts from the training corpus, computes every next-word probability for a
user-supplied word, and ranks the top three.

In [11]:
cond_corpus = [
    "I like NLP",
    "I like Python",
    "I like coffee",
    "I love NLP",
    "You like coffee",
]

cond_bigrams = {}
for sentence in cond_corpus:
    tokens = sentence.split()
    for w1, w2 in zip(tokens, tokens[1:]):
        cond_bigrams.setdefault(w1, {})
        cond_bigrams[w1][w2] = cond_bigrams[w1].get(w2, 0) + 1

def conditional_probabilities(word):
    candidates = cond_bigrams.get(word, {})
    total = sum(candidates.values())
    return {w: c / total for w, c in candidates.items()}

query_word = "like"
probs = conditional_probabilities(query_word)
ranked = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)

print("Possible next words:")
for w, p in ranked:
    print(f"{w} : {p:.2f}")

predicted = ranked[0][0]
print(f"\nPredicted next word: {predicted}")

print("\nTop Predictions")
for i, (w, p) in enumerate(ranked[:3], 1):
    print(f"{i}. {w} ({p:.2f})")

Possible next words:
coffee : 0.50
NLP : 0.25
Python : 0.25

Predicted next word: coffee

Top Predictions
1. coffee (0.50)
2. NLP (0.25)
3. Python (0.25)
